# Label Construction & Noise Analysis

This notebook defines the target label, analyzes label quality, identifies noise sources, and documents assumptions for the credit default prediction task.

**Key focus**: Understanding label quality and noise is critical for PD estimation, as mislabeled examples can bias probability estimates and degrade calibration.

In [ ]:
import pandas as pd
import numpy as np

# Load raw dataset
df = pd.read_csv('../data/raw/default_of_credit_card_clients.csv')

# Apply temporal splits from Step 2
train_ids = df[df['ID'] <= 20000]['ID'].values
val_ids = df[(df['ID'] > 20000) & (df['ID'] <= 25000)]['ID'].values
test_ids = df[df['ID'] > 25000]['ID'].values

train_df = df[df['ID'].isin(train_ids)].copy()
val_df = df[df['ID'].isin(val_ids)].copy()
test_df = df[df['ID'].isin(test_ids)].copy()

print(f"Train: {len(train_df)} samples")
print(f"Validation: {len(val_df)} samples")
print(f"Test: {len(test_df)} samples")

## Step 1: Label Definition

### Target Variable

**Column name**: `default payment next month`

**Type**: Binary (0 = no default, 1 = default)

**Formal definition**: 
- `Y = 1` if the borrower is recorded as defaulting in month 0 per dataset definition (the month following the observation window)
- `Y = 0` if the borrower is not recorded as defaulting in month 0

### Mapping to Outcome Window (from Step 2)

From the temporal framing established in `01_temporal_framing.ipynb`:

- **Observation window**: Months -6 to -1 (historical payment data available at decision time)
- **Decision point**: End of month -1
- **Outcome window**: Month 0 (the month immediately following the observation window)
- **Target**: `default payment next month` = outcome in month 0

**Temporal alignment**:
```
Month -6  -5  -4  -3  -2  -1  |  0
[Observation Window]          | [Outcome]
PAY_6 ... PAY_2 PAY_0         | default payment next month
```

The label captures default behavior in the **first month after the observation period**, not during the observation period itself.

### What "Default" Means in This Dataset

Based on the UCI dataset documentation and standard credit card default definitions:

**Default definition**: Borrower is recorded as defaulting in month 0 per the dataset definition.

**Operational interpretation** (based on standard credit card default definitions):
- The dataset labels a borrower as defaulting (`Y = 1`) if they fail to make minimum payment in month 0
- This is a single-month observation window for default classification

**Note**: This is a **single-month default definition**. It does not require:
- Multiple months of missed payments
- Charge-off status
- Account closure
- 30/60/90 days past due thresholds

This is a relatively **loose default definition** compared to industry standards (which often use 90+ days past due or charge-off).

In [ ]:
# Inspect target variable
target_col = 'default payment next month'

print("=== TARGET VARIABLE INSPECTION ===\n")
print(f"Column name: '{target_col}'")
print(f"Data type: {df[target_col].dtype}")
print(f"Unique values: {sorted(df[target_col].unique())}")
print(f"\nValue counts (full dataset):")
print(df[target_col].value_counts().sort_index())
print(f"\nProportions (full dataset):")
print(df[target_col].value_counts(normalize=True).sort_index())
print(f"\nMissing values: {df[target_col].isna().sum()}")

## Step 2: Analyze Label Quality

### Class Balance Across Splits

We examine the default rate (proportion of `Y = 1`) across train, validation, and test sets to check for:
- **Stability**: Similar default rates indicate consistent label distribution
- **Temporal shift**: Changes in default rate may indicate distribution shift over time
- **Split validity**: Large differences might indicate issues with the temporal proxy

In [ ]:
# Analyze class balance across splits
target_col = 'default payment next month'

print("=== CLASS BALANCE ACROSS SPLITS ===\n")

for split_name, split_df in [('Train', train_df), ('Validation', val_df), ('Test', test_df)]:
    default_rate = split_df[target_col].mean()
    default_count = split_df[target_col].sum()
    total = len(split_df)
    
    print(f"{split_name}:")
    print(f"  Total samples: {total:,}")
    print(f"  Defaults (Y=1): {default_count:,} ({default_rate:.4f} or {default_rate*100:.2f}%)")
    print(f"  Non-defaults (Y=0): {total - default_count:,} ({(1-default_rate):.4f} or {(1-default_rate)*100:.2f}%)")
    print()

# Compare default rates
train_rate = train_df[target_col].mean()
val_rate = val_df[target_col].mean()
test_rate = test_df[target_col].mean()

print("=== DEFAULT RATE COMPARISON ===\n")
print(f"Train default rate:      {train_rate:.4f} ({train_rate*100:.2f}%)")
print(f"Validation default rate: {val_rate:.4f} ({val_rate*100:.2f}%)")
print(f"Test default rate:       {test_rate:.4f} ({test_rate*100:.2f}%)")
print(f"\nDifference (Val - Train): {val_rate - train_rate:.4f} ({(val_rate - train_rate)*100:.2f} percentage points)")
print(f"Difference (Test - Train): {test_rate - train_rate:.4f} ({(test_rate - train_rate)*100:.2f} percentage points)")
print(f"Difference (Test - Val):   {test_rate - val_rate:.4f} ({(test_rate - val_rate)*100:.2f} percentage points)")

### Stability of Default Rate Over Temporal Split

**Interpretation**:
- **Stable rates** (within ~1-2 percentage points): Suggest consistent label distribution, supporting the ID-as-temporal-proxy assumption
- **Increasing rates**: May indicate deteriorating credit quality over time (economic downturn, portfolio expansion to riskier segments)
- **Decreasing rates**: May indicate improving credit quality or tighter underwriting over time
- **Large fluctuations** (>5 percentage points): May indicate issues with the temporal proxy or true distribution shift

**For PD estimation**: Stable default rates are preferable, as they suggest the underlying risk distribution is consistent. Large shifts require careful calibration adjustment.

### Edge Cases: Borderline Repayment Behavior

We examine borrowers with borderline payment behavior to identify potential label ambiguity:

1. **Borrowers who paid late but not defaulted** (Y=0): Recent payment delays (PAY_0 > 0) but made payment in month 0
2. **Borrowers who defaulted but had good history** (Y=1): Good payment history but defaulted in month 0
3. **Borrowers with inconsistent patterns**: Mixed payment behavior that makes default prediction ambiguous